# Phase 3 — Modeling

**Goal:** Train and evaluate regression models to predict birth weight (grams).  
**Success criteria:**  
- MAE < 300 g  
- R² > 0.70  
- LBW Sensitivity > 85 % (< 2500 g threshold)

**Flow:**
1. Setup & data loading  
2. Baseline (DummyRegressor)  
3. Candidate model screening (Ridge, Random Forest, LightGBM, XGBoost)  
4. Hyperparameter optimisation (Optuna, best two candidates)  
5. Test evaluation of best model  
6. Feature importance (permutation + SHAP)  
7. Model registration in MLflow

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Make project root importable when running the notebook directly
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings

import mlflow
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print(f"Project root: {ROOT}")
print(f"MLflow version: {mlflow.__version__}")

In [ ]:
from src.data.split import load_splits
from src.models.evaluate import evaluate_split
from src.models.pipeline import MODEL_FEATURES, build_pipeline, get_X_y
from src.models.train import EXPERIMENT_NAME, run_hpo, train_and_log

# MLflow tracking URI (local file store by default)
mlflow.set_tracking_uri(ROOT / "mlruns")
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 2. Load Splits

In [ ]:
PROCESSED_DIR = ROOT / "data" / "processed"

train_df, val_df, test_df = load_splits(PROCESSED_DIR)

print(f"Train : {len(train_df):>10,} rows  (year < 2010)")
print(f"Val   : {len(val_df):>10,} rows  (2010 ≤ year < 2016)")
print(f"Test  : {len(test_df):>10,} rows  (year ≥ 2016)")
print(f"Features available: {len([c for c in MODEL_FEATURES if c in train_df.columns])}/{len(MODEL_FEATURES)}")

In [ ]:
X_train, y_train = get_X_y(train_df)
X_val,   y_val   = get_X_y(val_df)
X_test,  y_test  = get_X_y(test_df)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_val  : {X_val.shape}  y_val  : {y_val.shape}")
print(f"X_test : {X_test.shape}  y_test : {y_test.shape}")
print(f"\nTarget stats (train):")
print(y_train.describe().round(1))
print(f"\nLBW prevalence (train): {(y_train < 2500).mean()*100:.2f}%")

## 3. Baseline — DummyRegressor (mean)

Establishes the floor: always predicting the training-set mean.

In [ ]:
from sklearn.dummy import DummyRegressor

baseline_pipe = build_pipeline(DummyRegressor(strategy="mean"))

baseline_metrics = train_and_log(
    pipeline=baseline_pipe,
    model_name="baseline_mean",
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    tags={"phase": "3", "stage": "baseline"},
)

print("Baseline validation metrics:")
for k, v in baseline_metrics.items():
    if k.startswith("val_"):
        print(f"  {k}: {v}")

## 4. Candidate Model Screening

Train each candidate with default hyperparameters on a 500 k-row sample to rank them quickly.

In [ ]:
# Sub-sample training data for fast screening (full training done in HPO step)
SCREEN_N = 500_000
rng = np.random.default_rng(42)
screen_idx = rng.choice(len(X_train), size=min(SCREEN_N, len(X_train)), replace=False)
X_screen = X_train.iloc[screen_idx]
y_screen = y_train.iloc[screen_idx]

print(f"Screening on {len(X_screen):,} rows")

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

CANDIDATES = {
    "ridge_default": (Ridge(alpha=1.0), True),         # scale_numeric=True for linear
    "lgbm_default":  (LGBMRegressor(n_estimators=300, random_state=42, verbose=-1), False),
    "xgb_default":   (XGBRegressor(n_estimators=300, random_state=42, verbosity=0), False),
}

screening_results = {}

for name, (model, scale) in CANDIDATES.items():
    print(f"\n--- {name} ---")
    pipe = build_pipeline(model, scale_numeric=scale)
    metrics = train_and_log(
        pipeline=pipe,
        model_name=name,
        X_train=X_screen,
        y_train=y_screen,
        X_val=X_val,
        y_val=y_val,
        tags={"phase": "3", "stage": "screening", "n_train": str(len(X_screen))},
    )
    screening_results[name] = metrics
    print(f"  val_mae={metrics['val_mae']:.1f}g  val_r2={metrics['val_r2']:.4f}  "
          f"lbw_sensitivity={metrics['val_sensitivity']:.3f}")

In [ ]:
# Screening summary table
summary_rows = []
for name, m in screening_results.items():
    summary_rows.append({
        "model": name,
        "val_mae": m["val_mae"],
        "val_rmse": m["val_rmse"],
        "val_r2": m["val_r2"],
        "val_lbw_sensitivity": m["val_sensitivity"],
    })

screen_df = pd.DataFrame(summary_rows).sort_values("val_mae")
print(screen_df.to_string(index=False))

## 5. Hyperparameter Optimisation (Optuna)

Run HPO on the top candidate(s) identified in the screening step. Using the full training set.

In [ ]:
# --- LightGBM HPO ---

def lgbm_pipeline_fn(trial):
    params = {
        "n_estimators":    trial.suggest_int("n_estimators", 200, 1000, step=100),
        "num_leaves":      trial.suggest_int("num_leaves", 31, 255),
        "learning_rate":   trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
        "subsample":       trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":       trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":      trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
    }
    return build_pipeline(
        LGBMRegressor(**params, random_state=42, verbose=-1),
        scale_numeric=False,
    )

lgbm_best_params, lgbm_study = run_hpo(
    model_name="lgbm_hpo",
    create_pipeline_fn=lgbm_pipeline_fn,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    n_trials=30,
)

print(f"\nBest LightGBM params:")
for k, v in lgbm_best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# --- XGBoost HPO ---

def xgb_pipeline_fn(trial):
    params = {
        "n_estimators":  trial.suggest_int("n_estimators", 200, 1000, step=100),
        "max_depth":     trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":     trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":     trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":    trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 50),
    }
    return build_pipeline(
        XGBRegressor(**params, random_state=42, verbosity=0, tree_method="hist"),
        scale_numeric=False,
    )

xgb_best_params, xgb_study = run_hpo(
    model_name="xgb_hpo",
    create_pipeline_fn=xgb_pipeline_fn,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    n_trials=30,
)

print(f"\nBest XGBoost params:")
for k, v in xgb_best_params.items():
    print(f"  {k}: {v}")

## 6. Train Best Models with Optimal Hyperparameters (Full Training Set)

In [ ]:
# Retrain LightGBM with best params on full training data
lgbm_best_pipe = build_pipeline(
    LGBMRegressor(**lgbm_best_params, random_state=42, verbose=-1),
    scale_numeric=False,
)

lgbm_final_metrics = train_and_log(
    pipeline=lgbm_best_pipe,
    model_name="lgbm_final",
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    params=lgbm_best_params,
    tags={"phase": "3", "stage": "final", "algo": "lightgbm"},
)

print("LightGBM final — validation:")
for k, v in lgbm_final_metrics.items():
    if k.startswith("val_"):
        print(f"  {k}: {v}")

In [ ]:
# Retrain XGBoost with best params on full training data
xgb_best_pipe = build_pipeline(
    XGBRegressor(**xgb_best_params, random_state=42, verbosity=0, tree_method="hist"),
    scale_numeric=False,
)

xgb_final_metrics = train_and_log(
    pipeline=xgb_best_pipe,
    model_name="xgb_final",
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    params=xgb_best_params,
    tags={"phase": "3", "stage": "final", "algo": "xgboost"},
)

print("XGBoost final — validation:")
for k, v in xgb_final_metrics.items():
    if k.startswith("val_"):
        print(f"  {k}: {v}")

In [ ]:
# Pick best model by val_mae
candidates = {
    "lgbm_final": (lgbm_best_pipe, lgbm_final_metrics),
    "xgb_final":  (xgb_best_pipe,  xgb_final_metrics),
}
best_name = min(candidates, key=lambda k: candidates[k][1]["val_mae"])
best_pipe, best_val_metrics = candidates[best_name]

print(f"\nBest model: {best_name}")
print(f"  val_mae         = {best_val_metrics['val_mae']:.1f} g  (target < 300 g)")
print(f"  val_r2          = {best_val_metrics['val_r2']:.4f}  (target > 0.70)")
print(f"  val_sensitivity = {best_val_metrics['val_sensitivity']:.3f}  (target > 0.85)")

## 7. Test Set Evaluation

**Run this only once** after the best model is selected. Test data must not influence model selection.

In [ ]:
test_metrics = evaluate_split(best_pipe, X_test, y_test, split_name="test")

print(f"\n{'='*50}")
print(f"FINAL TEST RESULTS ({best_name})")
print(f"{'='*50}")
print(f"  MAE            : {test_metrics['test_mae']:.1f} g  {'✓' if test_metrics['test_mae'] < 300 else '✗'} (target < 300 g)")
print(f"  RMSE           : {test_metrics['test_rmse']:.1f} g")
print(f"  R²             : {test_metrics['test_r2']:.4f}  {'✓' if test_metrics['test_r2'] > 0.70 else '✗'} (target > 0.70)")
print(f"  LBW Sensitivity: {test_metrics['test_sensitivity']:.3f}  {'✓' if test_metrics['test_sensitivity'] > 0.85 else '✗'} (target > 0.85)")
print(f"  LBW Specificity: {test_metrics['test_specificity']:.3f}")
print(f"  LBW Precision  : {test_metrics['test_precision']:.3f}")
print(f"  LBW F1         : {test_metrics['test_f1']:.3f}")
print(f"  LBW prevalence : {test_metrics['test_lbw_prevalence_pct']:.2f}%")

## 8. Feature Importance

In [ ]:
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

# Permutation importance on validation set (model-agnostic)
perm = permutation_importance(
    best_pipe, X_val, y_val,
    n_repeats=10, random_state=42, scoring="neg_mean_absolute_error",
    n_jobs=-1,
)

perm_df = pd.DataFrame(
    {"feature": X_val.columns, "importance": perm.importances_mean}
).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(perm_df["feature"][:15][::-1], perm_df["importance"][:15][::-1])
ax.set_xlabel("Mean decrease in MAE (permutation importance)")
ax.set_title(f"Top 15 Features — {best_name}")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "feature_importance_permutation.png", dpi=150)
plt.show()
print(perm_df.head(10).to_string(index=False))

In [ ]:
# SHAP values (sample 5k rows for speed)
try:
    import shap

    shap_sample = X_val.sample(5000, random_state=42)
    # Extract the final estimator for SHAP (TreeExplainer requires the raw model)
    final_model = best_pipe.named_steps["model"]
    preprocessor = best_pipe.named_steps["preprocessor"]
    X_shap = pd.DataFrame(
        preprocessor.transform(shap_sample),
        columns=MODEL_FEATURES,
    )

    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_shap)

    plt.figure(figsize=(8, 6))
    shap.summary_plot(shap_values, X_shap, plot_type="bar", show=False)
    plt.tight_layout()
    plt.savefig(ROOT / "reports" / "figures" / "shap_summary.png", dpi=150)
    plt.show()
    print("SHAP plot saved.")
except ImportError:
    print("shap not installed — run: pip install shap")
except Exception as e:
    print(f"SHAP failed: {e}")

## 9. Register Best Model in MLflow

In [ ]:
# Log test metrics to the final model's MLflow run, then register it
run_id = best_val_metrics["mlflow_run_id"]

with mlflow.start_run(run_id=run_id):
    mlflow.log_metrics(test_metrics)
    mlflow.set_tag("test_evaluated", "true")
    mlflow.set_tag("best_model", "true")

MODEL_REGISTRY_NAME = "stp-natality-birth-weight"
model_uri = f"runs:/{run_id}/model"

registered = mlflow.register_model(model_uri=model_uri, name=MODEL_REGISTRY_NAME)
print(f"Model registered: {MODEL_REGISTRY_NAME} v{registered.version}")
print(f"Run ID          : {run_id}")
print(f"Model URI       : {model_uri}")

In [ ]:
# Smoke test: load the registered model and predict on 5 rows
loaded_model = mlflow.sklearn.load_model(model_uri)
sample_preds = loaded_model.predict(X_test.head())
print("Sample predictions (grams):")
for i, (pred, true) in enumerate(zip(sample_preds, y_test.values[:5])):
    print(f"  Row {i}: predicted={pred:.0f}g  actual={true:.0f}g  error={abs(pred-true):.0f}g")

## 10. Summary

| Metric | Target | Result |
|--------|--------|--------|
| MAE | < 300 g | *(fill after run)* |
| R² | > 0.70 | *(fill after run)* |
| LBW Sensitivity | > 85% | *(fill after run)* |

**Next steps (Phase 4 — Evaluation & Deployment):**
- Create `docs/modeling_report.md` with full results and interpretation
- Set up a FastAPI inference endpoint (`src/api/`)
- Add drift monitoring with Evidently
- Build `notebooks/05_evaluation.ipynb` for residual analysis and fairness audit